# ⚡ Módulo 16 - Notebook 01: Caso Integrador Auditoría y Conciliación

## 📊 Proyecto End-to-End de Auditoría Automatizada

**Libro:** Saliendo de lo Pandito  
**Módulo:** 16 - Proyectos Integradores y GitHub  
**Duración estimada:** 90 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Desarrollar** un proyecto completo de auditoría  
✅ **Aplicar** técnicas de conciliación automatizada  
✅ **Detectar** inconsistencias y fraudes  
✅ **Generar** reportes ejecutivos  
✅ **Integrar** todos los conocimientos del curso

---

## 📋 Pre-requisitos

* ✅ Módulos 05-15 completados
* ✅ Conocimiento de Pandas y PySpark
* ✅ Familiaridad con procesos de auditoría

---

## 📚 Contenido

1. Caso de Negocio: Auditoría de Ventas
2. Carga y Validación de Datos
3. Conciliación de Registros
4. Detección de Anomalías
5. Reporte de Hallazgos
6. Dashboard Ejecutivo

---

## 💡 Por qué importa

**Auditoría automatizada = ahorro y precisión:**

* ⏱️ **Tiempo:** De semanas a horas
* 🎯 **Precisión:** 100% de cobertura
* 🔍 **Detección:** Patrones imposibles manualmente
* 💰 **ROI:** Fraudes detectados valen millones

**Proyecto portfolio para conseguir trabajo**

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

print("💾 CARGANDO DATOS PARA AUDITORÍA")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar datos de ventas de Los Andes Market
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    print(f"\n✅ Datos de auditoría cargados")
    print(f"   📊 Registros de ventas: {len(df_ventas):,}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   🏪 Sucursales: {df_ventas['sucursal_id'].nunique()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Calcular métricas para auditoría
    print(f"\n📊 Resumen para auditoría:")
    print(f"   • Ventas totales: ${df_ventas['ventas'].sum():,.2f}")
    print(f"   • Ticket promedio: ${df_ventas['ventas'].mean():,.2f}")
    print(f"   • Transacciones: {len(df_ventas):,}")
    
    # Verificar integridad de datos
    print(f"\n🔍 Verificación de integridad:")
    nulls_ventas = df_ventas['ventas'].isnull().sum()
    nulls_fecha = df_ventas['fecha'].isnull().sum()
    duplicados = df_ventas.duplicated().sum()
    
    print(f"   • Nulos en ventas: {nulls_ventas} ({100*nulls_ventas/len(df_ventas):.2f}%)")
    print(f"   • Nulos en fecha: {nulls_fecha} ({100*nulls_fecha/len(df_ventas):.2f}%)")
    print(f"   • Duplicados: {duplicados} ({100*duplicados/len(df_ventas):.2f}%)")
    
    if nulls_ventas == 0 and nulls_fecha == 0 and duplicados == 0:
        print(f"   ✅ Datos limpios (sin nulos ni duplicados)")
    else:
        print(f"   ⚠️ Se detectaron problemas de calidad")
    
    print(f"\n🎯 Este proyecto incluirá:")
    print(f"   1. Conciliación de ventas por sucursal")
    print(f"   2. Detección de anomalías y outliers")
    print(f"   3. Análisis de tendencias sospechosas")
    print(f"   4. Reporte ejecutivo de hallazgos")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Auditoría Automatizada con Datos

### 🔍 ¿Qué es Auditoría Automatizada?

**Auditoría Automatizada:** Uso de herramientas de análisis de datos para verificar registros y detectar inconsistencias.

**Ventajas vs Auditoría Manual:**

| Manual | Automatizada |
|--------|-------------|
| Muestreo (5-10%) | Cobertura 100% |
| Días/Semanas | Horas |
| Propenso a errores | Preciso y reproducible |
| Reactivo | Proactivo |

---

### 📊 Tipos de Auditoría con Datos

**1️⃣ Conciliación de Registros**

Comparar dos fuentes de datos que deberían coincidir.

```python
# Ejemplo: Ventas declaradas vs registradas
df_declaradas = ...
df_registradas = ...

# Identificar diferencias
diferencias = pd.merge(
    df_declaradas, 
    df_registradas, 
    on='transaccion_id', 
    how='outer', 
    indicator=True
)

faltantes = diferencias[diferencias['_merge'] != 'both']
```

---

**2️⃣ Detección de Anomalías**

Encontrar transacciones que no siguen patrones normales.

```python
# Ejemplo: Ventas fuera de rango normal
mean = df['ventas'].mean()
std = df['ventas'].std()

# Z-score > 3 = anomalía
df['z_score'] = (df['ventas'] - mean) / std
anomalias = df[abs(df['z_score']) > 3]
```

---

**3️⃣ Análisis de Ley de Benford**

**Ley de Benford:** En datos naturales, el primer dígito sigue una distribución predecible.

**Distribución esperada:**
```
Dígito | Frecuencia esperada
   1   | 30.1%
   2   | 17.6%
   3   | 12.5%
   ...
   9   | 4.6%
```

**Uso:** Detectar fraude en datos contables.

```python
# Extraer primer dígito
df['primer_digito'] = df['ventas'].astype(str).str[0].astype(int)

# Comparar con Benford
frecuencias_observadas = df['primer_digito'].value_counts(normalize=True)
frecuencias_esperadas = [0.301, 0.176, 0.125, 0.097, 0.079, 0.067, 0.058, 0.051, 0.046]

# Chi-square test
from scipy.stats import chisquare
chi2, p_value = chisquare(frecuencias_observadas, frecuencias_esperadas)

if p_value < 0.05:
    print("⚠️ Posible manipulación de datos detectada")
```

---

**4️⃣ Análisis de Duplicados**

Encontrar transacciones duplicadas (posible fraude).

```python
# Duplicados exactos
duplicados_exactos = df[df.duplicated(keep=False)]

# Duplicados "fuzzy" (similar pero no exacto)
from fuzzywuzzy import fuzz

# Comparar nombres de clientes
for i, row1 in df.iterrows():
    for j, row2 in df.iterrows():
        if i < j:
            similarity = fuzz.ratio(row1['cliente'], row2['cliente'])
            if similarity > 90:  # >90% similar
                print(f"Posible duplicado: {row1['cliente']} vs {row2['cliente']}")
```

---

### 🎯 Caso de Uso: Auditoría de Ventas

**Escenario:**
Auditar ventas de Los Andes Market para detectar:

1. **Diferencias** entre ventas declaradas vs registradas
2. **Anomalías** (transacciones atípicas)
3. **Patrones sospechosos** (Benford, duplicados)
4. **Tendencias** (caídas o subidas abruptas)

---

### 📈 Pipeline de Auditoría

**Flujo:**

```
1. CARGA
   ↓ Cargar datos de múltiples fuentes
   
2. VALIDACIÓN
   ↓ Verificar integridad (nulos, duplicados, tipos)
   
3. CONCILIACIÓN
   ↓ Comparar fuentes (declarado vs registrado)
   
4. ANÁLISIS
   ↓ Detectar anomalías, aplicar Benford
   
5. REPORTE
   ↓ Generar hallazgos y visualizaciones
   
6. ACCIÓN
   ↓ Investigar casos sospechosos
```

---

### 🚨 Red Flags Comunes

**Señales de alerta:**

* 🔴 **Round numbers:** Muchas transacciones en números redondos ($1000, $5000)
* 🔴 **Benford violation:** Distribución anormal de primeros dígitos
* 🔴 **Duplicados:** Misma transacción múltiples veces
* 🔴 **Outliers extremos:** Ventas 10x fuera del rango normal
* 🔴 **Patrones temporales:** Picos sospechosos en fechas específicas
* 🔴 **Missing data:** Falta de registros en períodos críticos

---

### 💼 Herramientas del Auditor de Datos

**Stack tecnológico:**

* **Pandas:** Manipulación y análisis
* **NumPy:** Cálculos estadísticos
* **SciPy:** Tests estadísticos (chi-square, t-test)
* **Plotly:** Visualizaciones interactivas
* **PySpark:** Escala a millones de transacciones

---

### 📊 Reporte de Hallazgos

**Estructura típica:**

```markdown
# REPORTE DE AUDITORÍA
## Los Andes Market - Q1 2024

### RESUMEN EJECUTIVO
- Transacciones auditadas: 10,000
- Hallazgos críticos: 5
- Hallazgos moderados: 12
- Riesgo: MEDIO

### HALLAZGOS CRÍTICOS

1. **Duplicados (5 casos)**
   - Descripción: Transacciones idénticas en mismo día
   - Monto: $50,000
   - Acción: Investigar sucursal SUC001

2. **Violación Benford**
   - Descripción: Distribución anormal de dígitos
   - Sucursales: SUC003, SUC007
   - Acción: Auditoría profunda

...

### RECOMENDACIONES
1. Implementar validación automática de duplicados
2. Revisar controles internos en SUC001
3. Capacitar personal en prevención de fraudes
```

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("📊 PROYECTO INTEGRADOR: AUDITORÍA AUTOMATIZADA")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n🎯 En este proyecto aplicarás:")
print("  • Conciliación de registros")
print("  • Detección de anomalías (Z-score, IQR)")
print("  • Ley de Benford para fraude")
print("  • Análisis de duplicados")
print("  • Reportes ejecutivos")

print("\n📖 Técnicas clave:")
print("  - pd.merge() con indicator para conciliación")
print("  - Z-score: (x - mean) / std")
print("  - IQR: Q3 - Q1 para outliers")
print("  - Chi-square test para Benford")
print("  - df.duplicated() para duplicados")

print("\n🔍 Herramientas de auditoría:")
print("  - Pandas: Manipulación de datos")
print("  - NumPy: Estadística")
print("  - SciPy: Tests estadísticos")
print("  - Plotly/Matplotlib: Visualizaciones")

print("\n" + "="*70)
print("✅ Listo para auditoría automatizada")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')